In [1]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Annotated, Literal, Optional
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.types import Send
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import operator
from pathlib import Path
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
class Task(BaseModel):
    id : int
    title: str
    brief : str = Field(..., description= 'What to cover ')


class Plan(BaseModel):
    blog_title : str
    tasks : List[Task]
    
class BlogState(TypedDict):
    topic: str
    plan: Plan
    # reducer: results from workers get concatenated automatically
    sections: Annotated[List[str], operator.add]
    final: str
output =  llm.with_structured_output(Plan).invoke('create a 2 to 3 sections on the gen ai topic')
output
Plan(blog_title='Gen AI', tasks=[Task(id=1, title='Introduction to Gen AI', brief='What is Gen AI and its applications'), Task(id=2, title='Benefits and Challenges of Gen AI', brief='Advantages and disadvantages of using Gen AI'), Task(id=3, title='Future of Gen AI', brief='Potential developments and impact of Gen AI on society')])
def orchestrator(state: BlogState) -> dict:
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan": plan}




def fanout(state: BlogState):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]

# here fanout send a list for each of the task stored in the plan



def worker(payload: dict) -> dict:
    topic = payload["topic"]
    plan  = payload['plan']
    task = payload['task']

    blog_title = plan.blog_title
    
    section_md = llm.invoke(
        [
            SystemMessage(content="Write one clean Markdown section."),
            HumanMessage(
                content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief: {task.brief}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}
     



def reducer(state: BlogState):
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    # ---- save to file ----
    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}


 
graph = StateGraph(BlogState)

graph.add_node('orchestrator', orchestrator)
graph.add_node('worker', worker)
graph.add_node('reducer', reducer)


graph.add_edge(START, 'orchestrator')
graph.add_conditional_edges('orchestrator', fanout, ['worker'])
graph.add_edge('worker', 'reducer')
graph.add_edge('reducer', END)


workflow = graph.compile()
workflow

out = workflow.invoke({"topic": "Write a blog on impact of ai on the environment", "sections": []})
out
{'topic': 'Write a blog on impact of ai on the environment',
 'plan': Plan(blog_title='The Impact of AI on the Environment', tasks=[Task(id=1, title='Introduction to AI and Environment', brief='Introduce the topic of AI and its growing impact on the environment'), Task(id=2, title='AI and Energy Consumption', brief='Discuss the energy consumption of AI systems and its effects on the environment'), Task(id=3, title='AI and Resource Utilization', brief='Examine how AI can optimize resource utilization and reduce waste'), Task(id=4, title='AI and Climate Change', brief='Explore the role of AI in monitoring and mitigating climate change'), Task(id=5, title='AI and Sustainable Development', brief='Discuss the potential of AI in achieving sustainable development goals'), Task(id=6, title='Challenges and Limitations', brief='Highlight the challenges and limitations of AI in environmental applications'), Task(id=7, title='Conclusion and Future Directions', brief='Summarize the impact of AI on the environment and outline future research directions')]),
 'sections': ['### Introduction to AI and Environment\nThe rapid advancement and integration of Artificial Intelligence (AI) into various aspects of our lives have sparked intense discussions about its potential impacts. One crucial area of concern is the environment. As AI technologies continue to evolve and expand, their effects on the planet are becoming increasingly significant. From energy consumption and e-waste generation to potential solutions for environmental challenges, the relationship between AI and the environment is complex and multifaceted. This blog aims to explore the growing impact of AI on the environment, delving into both the negative consequences and the promising opportunities that AI presents for environmental sustainability.',
  "### AI and Energy Consumption\nThe increasing use of Artificial Intelligence (AI) has significant implications for energy consumption and the environment. Training large AI models requires massive amounts of computational power, which in turn, consumes substantial amounts of energy. The energy consumption of AI systems is primarily due to the following factors:\n* **Computational Power**: The processing power required to train and run AI models is extremely high, leading to increased energy consumption.\n* **Data Centers**: The data centers that house AI systems require significant amounts of energy to power the servers, cooling systems, and other infrastructure.\n* **Networking**: The transmission of data between devices and data centers also contributes to energy consumption.\n\nThe effects of AI's energy consumption on the environment are multifaceted:\n* **Greenhouse Gas Emissions**: The production of energy required to power AI systems contributes to greenhouse gas emissions, exacerbating climate change.\n* **E-Waste**: The rapid obsolescence of AI hardware leads to a significant amount of electronic waste, which can harm the environment if not disposed of properly.\n* **Resource Depletion**: The extraction of rare earth minerals and other resources required for AI hardware can lead to environmental degradation and resource depletion.\n\nTo mitigate the environmental impact of AI's energy consumption, it is essential to develop more energy-efficient AI systems, invest in renewable energy sources, and implement sustainable practices throughout the AI lifecycle.",
  '### AI and Resource Utilization\nArtificial intelligence (AI) has the potential to significantly impact the environment by optimizing resource utilization and reducing waste. One of the primary ways AI can achieve this is through predictive maintenance, which involves using machine learning algorithms to predict when equipment or machinery is likely to fail. This allows for proactive maintenance, reducing the likelihood of unexpected failures and subsequent waste. Additionally, AI can be used to optimize resource allocation in various industries, such as manufacturing and logistics, by analyzing data and identifying areas where resources can be used more efficiently. For example, AI can help reduce energy consumption by optimizing energy usage in buildings and data centers. Furthermore, AI-powered systems can help reduce food waste by predicting food demand and optimizing supply chains. Overall, the use of AI in resource utilization can lead to significant reductions in waste and emissions, contributing to a more sustainable environment.',
  '### AI and Climate Change\nArtificial intelligence (AI) is increasingly being used to monitor and mitigate the effects of climate change. One of the key applications of AI in this area is in the analysis of large datasets related to climate patterns, weather forecasting, and environmental monitoring. AI algorithms can quickly process vast amounts of data from various sources, including satellite imagery, sensors, and weather stations, to identify trends and patterns that may not be apparent to human analysts. This information can be used to predict climate-related events, such as hurricanes, droughts, and heatwaves, allowing for more effective planning and response. Additionally, AI can be used to optimize energy consumption and reduce greenhouse gas emissions by identifying areas of inefficiency and providing recommendations for improvement. For example, AI-powered smart grids can optimize energy distribution and predict energy demand, while AI-powered building management systems can optimize heating, cooling, and lighting usage. Overall, the use of AI in climate change mitigation has the potential to make a significant impact, and its applications are likely to continue to grow in the coming years.',
  "### AI and Sustainable Development\nThe potential of Artificial Intelligence (AI) in achieving sustainable development goals is vast and multifaceted. AI can be leveraged to optimize resource utilization, reduce waste, and promote eco-friendly practices. For instance, AI-powered sensors can be used to monitor and manage energy consumption in buildings, while machine learning algorithms can help predict and prevent natural disasters such as floods and wildfires. Additionally, AI can facilitate sustainable agriculture by analyzing soil conditions, weather patterns, and crop yields to optimize farming practices. Furthermore, AI-driven systems can help reduce carbon emissions by streamlining transportation systems, optimizing supply chains, and promoting the use of renewable energy sources. By harnessing the power of AI, we can accelerate our progress towards achieving the United Nations' Sustainable Development Goals (SDGs) and create a more sustainable future for generations to come.",
  '### Challenges and Limitations\nThe integration of AI in environmental applications is not without its challenges and limitations. One of the primary concerns is the significant amount of data required to train AI models, which can be difficult to obtain, particularly in remote or hard-to-reach areas. Additionally, the quality of the data can be a major issue, with factors such as sensor accuracy and data preprocessing affecting the accuracy of the models. \nAnother challenge is the lack of standardization in AI applications, making it difficult to compare and integrate different models and systems. Furthermore, the high computational power required to run AI models can lead to increased energy consumption, which can have a negative impact on the environment. \nThere are also concerns about the potential for AI to displace human workers in environmental fields, such as conservation and sustainability. Moreover, the development and deployment of AI systems can be costly, which can limit their adoption in developing countries or in areas with limited resources. \nFinally, there is a need for more research into the long-term effects of AI on the environment, as well as the potential risks and unintended consequences of relying on AI systems to manage and protect the environment.',
  '### Conclusion and Future Directions\nThe impact of AI on the environment is a complex and multifaceted issue. On one hand, AI has the potential to significantly reduce greenhouse gas emissions and mitigate the effects of climate change by optimizing energy consumption, improving resource allocation, and enhancing renewable energy sources. On the other hand, the production and deployment of AI systems can result in significant environmental costs, including e-waste, energy consumption, and resource extraction. \nTo fully realize the potential of AI to benefit the environment, future research should focus on developing more sustainable AI systems, improving the energy efficiency of AI algorithms, and exploring new applications of AI in environmental sustainability. Additionally, there is a need for more comprehensive and nuanced assessments of the environmental impacts of AI, as well as the development of standards and guidelines for the responsible development and deployment of AI systems. \nUltimately, the future of AI and the environment will depend on our ability to balance the benefits of AI with its potential environmental costs, and to prioritize the development of AI systems that are not only intelligent, but also sustainable and responsible.'],
 'final': "# The Impact of AI on the Environment\n\n### Introduction to AI and Environment\nThe rapid advancement and integration of Artificial Intelligence (AI) into various aspects of our lives have sparked intense discussions about its potential impacts. One crucial area of concern is the environment. As AI technologies continue to evolve and expand, their effects on the planet are becoming increasingly significant. From energy consumption and e-waste generation to potential solutions for environmental challenges, the relationship between AI and the environment is complex and multifaceted. This blog aims to explore the growing impact of AI on the environment, delving into both the negative consequences and the promising opportunities that AI presents for environmental sustainability.\n\n### AI and Energy Consumption\nThe increasing use of Artificial Intelligence (AI) has significant implications for energy consumption and the environment. Training large AI models requires massive amounts of computational power, which in turn, consumes substantial amounts of energy. The energy consumption of AI systems is primarily due to the following factors:\n* **Computational Power**: The processing power required to train and run AI models is extremely high, leading to increased energy consumption.\n* **Data Centers**: The data centers that house AI systems require significant amounts of energy to power the servers, cooling systems, and other infrastructure.\n* **Networking**: The transmission of data between devices and data centers also contributes to energy consumption.\n\nThe effects of AI's energy consumption on the environment are multifaceted:\n* **Greenhouse Gas Emissions**: The production of energy required to power AI systems contributes to greenhouse gas emissions, exacerbating climate change.\n* **E-Waste**: The rapid obsolescence of AI hardware leads to a significant amount of electronic waste, which can harm the environment if not disposed of properly.\n* **Resource Depletion**: The extraction of rare earth minerals and other resources required for AI hardware can lead to environmental degradation and resource depletion.\n\nTo mitigate the environmental impact of AI's energy consumption, it is essential to develop more energy-efficient AI systems, invest in renewable energy sources, and implement sustainable practices throughout the AI lifecycle.\n\n### AI and Resource Utilization\nArtificial intelligence (AI) has the potential to significantly impact the environment by optimizing resource utilization and reducing waste. One of the primary ways AI can achieve this is through predictive maintenance, which involves using machine learning algorithms to predict when equipment or machinery is likely to fail. This allows for proactive maintenance, reducing the likelihood of unexpected failures and subsequent waste. Additionally, AI can be used to optimize resource allocation in various industries, such as manufacturing and logistics, by analyzing data and identifying areas where resources can be used more efficiently. For example, AI can help reduce energy consumption by optimizing energy usage in buildings and data centers. Furthermore, AI-powered systems can help reduce food waste by predicting food demand and optimizing supply chains. Overall, the use of AI in resource utilization can lead to significant reductions in waste and emissions, contributing to a more sustainable environment.\n\n### AI and Climate Change\nArtificial intelligence (AI) is increasingly being used to monitor and mitigate the effects of climate change. One of the key applications of AI in this area is in the analysis of large datasets related to climate patterns, weather forecasting, and environmental monitoring. AI algorithms can quickly process vast amounts of data from various sources, including satellite imagery, sensors, and weather stations, to identify trends and patterns that may not be apparent to human analysts. This information can be used to predict climate-related events, such as hurricanes, droughts, and heatwaves, allowing for more effective planning and response. Additionally, AI can be used to optimize energy consumption and reduce greenhouse gas emissions by identifying areas of inefficiency and providing recommendations for improvement. For example, AI-powered smart grids can optimize energy distribution and predict energy demand, while AI-powered building management systems can optimize heating, cooling, and lighting usage. Overall, the use of AI in climate change mitigation has the potential to make a significant impact, and its applications are likely to continue to grow in the coming years.\n\n### AI and Sustainable Development\nThe potential of Artificial Intelligence (AI) in achieving sustainable development goals is vast and multifaceted. AI can be leveraged to optimize resource utilization, reduce waste, and promote eco-friendly practices. For instance, AI-powered sensors can be used to monitor and manage energy consumption in buildings, while machine learning algorithms can help predict and prevent natural disasters such as floods and wildfires. Additionally, AI can facilitate sustainable agriculture by analyzing soil conditions, weather patterns, and crop yields to optimize farming practices. Furthermore, AI-driven systems can help reduce carbon emissions by streamlining transportation systems, optimizing supply chains, and promoting the use of renewable energy sources. By harnessing the power of AI, we can accelerate our progress towards achieving the United Nations' Sustainable Development Goals (SDGs) and create a more sustainable future for generations to come.\n\n### Challenges and Limitations\nThe integration of AI in environmental applications is not without its challenges and limitations. One of the primary concerns is the significant amount of data required to train AI models, which can be difficult to obtain, particularly in remote or hard-to-reach areas. Additionally, the quality of the data can be a major issue, with factors such as sensor accuracy and data preprocessing affecting the accuracy of the models. \nAnother challenge is the lack of standardization in AI applications, making it difficult to compare and integrate different models and systems. Furthermore, the high computational power required to run AI models can lead to increased energy consumption, which can have a negative impact on the environment. \nThere are also concerns about the potential for AI to displace human workers in environmental fields, such as conservation and sustainability. Moreover, the development and deployment of AI systems can be costly, which can limit their adoption in developing countries or in areas with limited resources. \nFinally, there is a need for more research into the long-term effects of AI on the environment, as well as the potential risks and unintended consequences of relying on AI systems to manage and protect the environment.\n\n### Conclusion and Future Directions\nThe impact of AI on the environment is a complex and multifaceted issue. On one hand, AI has the potential to significantly reduce greenhouse gas emissions and mitigate the effects of climate change by optimizing energy consumption, improving resource allocation, and enhancing renewable energy sources. On the other hand, the production and deployment of AI systems can result in significant environmental costs, including e-waste, energy consumption, and resource extraction. \nTo fully realize the potential of AI to benefit the environment, future research should focus on developing more sustainable AI systems, improving the energy efficiency of AI algorithms, and exploring new applications of AI in environmental sustainability. Additionally, there is a need for more comprehensive and nuanced assessments of the environmental impacts of AI, as well as the development of standards and guidelines for the responsible development and deployment of AI systems. \nUltimately, the future of AI and the environment will depend on our ability to balance the benefits of AI with its potential environmental costs, and to prioritize the development of AI systems that are not only intelligent, but also sustainable and responsible.\n"}

{'topic': 'Write a blog on impact of ai on the environment',
 'plan': Plan(blog_title='The Impact of AI on the Environment', tasks=[Task(id=1, title='Introduction to AI and Environment', brief='Introduce the topic of AI and its growing impact on the environment'), Task(id=2, title='AI and Energy Consumption', brief='Discuss the energy consumption of AI systems and its effects on the environment'), Task(id=3, title='AI and Resource Utilization', brief='Examine how AI can optimize resource utilization and reduce waste'), Task(id=4, title='AI and Climate Change', brief='Explore the role of AI in monitoring and mitigating climate change'), Task(id=5, title='AI and Sustainable Development', brief='Discuss the potential of AI in achieving sustainable development goals'), Task(id=6, title='Challenges and Limitations', brief='Highlight the challenges and limitations of AI in environmental applications'), Task(id=7, title='Conclusion and Future Directions', brief='Summarize the impact of AI on 